# 2. Entrenamiento: RLCD con Jev como profesor

`entrenar.py` sigue la receta del [notebook oficial de Laya](https://github.com/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb),
en una sola GPU y sobre el checkpoint multilingüe:

1. **Casos** de `data/sintetico.csv`.
2. **Profesor.** Jev responde las mismas tres preguntas sobre cada caso (US$0,04 por 1.000; se guarda en
   `data/profesor.jsonl` y solo se piden los nuevos). Su distribución suaviza el objetivo (30 %) y los casos que no ve
   en la categoría pedida se descartan. Sin llave, o con `--sin-profesor`, la etiqueta se suaviza al 90 %.
3. **Reparto** 80 / 10 / 10: entrenamiento, calibración y validación. Los 20 tickets de prueba nunca entran.
4. **RLCD:** en cada paso se sacan 4 versiones con ruido de cada distribución, se premian con reglas de puntuación
   propias (log, esférica y RPS para `score`) y se suma la entropía cruzada suave. Se guarda la mejor época según la
   validación.
5. **Calibración:** una temperatura por tipo de pregunta con los casos apartados (`laya-multilingual` viene sin
   calibrar).
6. **Comparación** base frente a ajustada, y **guardado** en formato de checkpoint de Laya.

| Perfil | Casos | Épocas | Qué se entrena | GPU |
|---|---|---|---|---|
| `prueba` (por defecto) | 2 por combinación (72) | 1 | 6 capas superiores + cabeza (~45 M) | tope duro de 3 GB |
| `--completo` | todos | 4 (`--epocas`) | todo el modelo (322 M) | ~6 GB, pensada para 12 GB o más |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import modelo  # Antes que torch: caché de Hugging Face en el proyecto y llaves locales.

import torch

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Sin GPU: entrenar necesita CUDA")

## Prueba corta (≤ 3 GB de GPU)

Comprueba todo el circuito en una laptop; con 72 casos las cifras no dicen si el ajuste funciona.

In [ ]:
!cd {ROOT} && {sys.executable} entrenar.py

## Entrenamiento completo

En una RTX 4070 Ti SUPER, cada época sobre ~12.000 casos tarda unos minutos y usa ~6 GB.

In [ ]:
!cd {ROOT} && {sys.executable} entrenar.py --completo --salida .model-cache/laya-mesa-de-ayuda

## Resultado guardado junto al checkpoint

In [ ]:
import json
import evaluar

report = json.loads((ROOT / ".model-cache" / "laya-mesa-de-ayuda" / "resultados.json").read_text())
print({k: report[k] for k in ("perfil", "casos", "mejor_ce_validacion", "vram_max_gb")})
print(evaluar.table({"Validación: base": report["validacion"]["base"], "Validación: ajustada": report["validacion"]["ajustada"],
                     "Prueba: base": report["prueba"]["base"], "Prueba: ajustada": report["prueba"]["ajustada"]}))